# 📊 MarkMyBench: Interactive RAG vs PageIndex Evaluation

This notebook allows you to interactively run the benchmarking suite, evaluate the results using RAGAS, and visualize the performance differences between the **Traditional RAG** and **PageIndex (Vectorless RAG)** pipelines.

## 🛠 1. Setup
First, we need to ensure the environment variables are loaded and the package is in the Python path.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# Ensure the project root is in sys.path so we can import the rag_vs_pageindex module
project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load environment variables from the submodule directory
env_path = project_root / "rag_vs_pageindex" / ".env"
load_dotenv(dotenv_path=env_path)

if not os.environ.get("GOOGLE_API_KEY"):
    print("❌ ERROR: GOOGLE_API_KEY not found in .env file!")
else:
    print("✅ Environment loaded successfully.")

✅ Environment loaded successfully.


## 🧪 2. Run Benchmark

We can now run the benchmark on a specific document. 

> **Note on Rate Limits & Optimal Performance:**
> - **Gemini Free Tier:** Most users are on the free tier which has a hard limit of **15 RPM (Requests Per Minute)**.
> - **RAGAS Overhead:** Evaluating a single question with 4 metrics can take 6-10 separate LLM calls. At 15 RPM, you can only safely evaluate about **1-2 questions per minute**.
> - **Fix Applied:** I have set `max_workers=1` in the code, which forces sequential processing. If you still see 429 errors, consider a Paid tier or reducing the number of evaluation questions.
> - **PDF Syntax Errors:** If a PDF fails with `No /Root object`, ensure it is a valid PDF and not an HTML error page.

In [ ]:
from rag_vs_pageindex.evaluate import run_benchmark

# Specify your target document (URL or local path)
doc_source = "data/rag_vs_page_index/nist-cybersecurity-frameworks.pdf"

# Run the benchmark
# This will ingest the doc, build both indices, retrieve context, generate answers,
# and run RAGAS evaluation (Faithfulness, Precision, Recall, Relevancy).
df_results = run_benchmark(doc_source)

print("\n--- Evaluation Complete ---")

## 📈 3. Analyze Results

The results are stored in a DataFrame. We can now look at the mean scores for each pipeline.

In [ ]:
# Display the mean metrics per pipeline
numeric_cols = df_results.select_dtypes(include="number").columns.tolist()
summary = df_results.groupby("pipeline")[numeric_cols].mean()
summary

### Visualizing Latency vs Faithfulness
We can use `matplotlib` or `seaborn` to visualize the trade-offs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(data=df_results, x="pipeline", y="faithfulness", hue="pipeline")
plt.title("Comparison: Faithfulness (Higher is Better)")
plt.ylim(0, 1.1)
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=df_results, x="pipeline", y="latency_seconds", hue="pipeline")
plt.title("Comparison: Latency (Lower is Better)")
plt.show()

## 📄 4. Detailed View
You can inspect the full answers and retrieved contexts for specific questions.

In [ ]:
# Show first 2 rows of the raw results
df_results.head(2)